# FIS Mamdani — Suporte à Decisão Agrícola com Lógica Fuzzy

Sistema de inferência fuzzy (método **Mamdani**) que analisa variáveis climáticas para recomendar janelas de pulverização, monitorar estresse hídrico, sugerir irrigação e estimar produtividade.

## Instalação da dependência

In [4]:
pip install scikit-fuzzy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importação das bibliotecas

- **NumPy** — criação dos universos de discurso (intervalos numéricos das variáveis).
- **scikit-fuzzy** — motor de inferência fuzzy; `control` fornece as abstrações `Antecedent`, `Consequent` e `Rule`.

In [5]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

## Variáveis antecedentes (entradas)

Cada antecedente representa uma grandeza climática com seu respectivo universo de discurso:

| Variável | Faixa | Unidade |
|---|---|---|
| Temperatura | 0 – 60 | °C |
| Umidade | 0 – 100 | % |
| Chuva | 0 – 500 | mm |
| Vento | 0 – 150 | km/h |
| Delta T | 0 – 40 | °C |

In [6]:
temperature = ctrl.Antecedent(np.arange(0, 60, 0.1), 'Temperatura')
humidity = ctrl.Antecedent(np.arange(0, 101, 1.0), 'Umidade')
rain = ctrl.Antecedent(np.arange(0, 500, 0.2), 'Chuva')
wind = ctrl.Antecedent(np.arange(0, 150, 0.5), 'Vento')
delta_t = ctrl.Antecedent(np.arange(0, 40, 0.1), 'Delta T')

## Funções de pertinência dos antecedentes

Cada variável de entrada recebe **7 conjuntos fuzzy** gerados automaticamente via `automf`. Os nomes refletem termos agronômicos para facilitar a leitura das regras:

- **Temperatura**: frio_extremo, frio, ameno, **ideal**, quente, estresse_termico, critico
- **Umidade**: deserto, critico_seco, seco, **ideal**, umido, saturado, condensacao
- **Chuva**: seco, trace, leve, moderada, forte, muito_forte, extrema
- **Vento**: calmo, ideal_pulv, moderado, atencao_pulv, forte, muito_forte, tempestade
- **Delta T**: inversao_termica, ideal_pulv, aceitavel, atencao, critico_pulv, proibido, extremo

In [7]:
temperature.automf(
    number=7,
    names=['frio_extremo', 'frio', 'ameno', 'ideal', 'quente', 'estresse_termico', 'critico']
)
humidity.automf(
    number=7,
    names=['deserto', 'critico_seco', 'seco', 'ideal', 'umido', 'saturado', 'condensacao']
)
rain.automf(
    number=7,
    names=['seco', 'trace', 'leve', 'moderada', 'forte', 'muito_forte', 'extrema']
)
wind.automf(
    number=7,
    names=['calmo', 'ideal_pulv', 'moderado', 'atencao_pulv', 'forte', 'muito_forte', 'tempestade']
)
delta_t.automf(
    number=7,
    names=['inversao_termica', 'ideal_pulv', 'aceitavel', 'atencao', 'critico_pulv', 'proibido', 'extremo']
)

## Variáveis consequentes (saídas)

| Consequente | Universo | Descrição |
|---|---|---|
| Recomendação de Pulverização (`sp`) | 0 – 10 | Classifica em **proibida**, **atenção** ou **janela disponível** (funções triangulares manuais) |
| Estresse Hídrico (`wh`) | 0 – 1 | Índice normalizado de estresse hídrico da cultura |
| Recomendação de Irrigação (`ir`) | 0 – 10 | Nível de urgência para irrigação |
| Produtividade Estimada (`bp`) | 0 – 100 | Percentual estimado da produtividade potencial |

In [8]:
spray_recomendation = ctrl.Consequent(np.arange(0, 11, 1), 'sp')
spray_recomendation['proibida'] = fuzz.trimf(spray_recomendation.universe, [0, 0, 4])
spray_recomendation['atencao'] = fuzz.trimf(spray_recomendation.universe, [3, 5, 7])
spray_recomendation['janela_disponivel'] = fuzz.trimf(spray_recomendation.universe, [6, 10, 10])

water_stress = ctrl.Consequent(np.arange(0, 1.1, 0.1), 'wh')
water_stress['baixo'] = fuzz.trimf(water_stress.universe, [0, 0.2, 1])
water_stress['medio'] = fuzz.trimf(water_stress.universe, [0, 0.5, 1])
water_stress['alto'] = fuzz.trimf(water_stress.universe, [0.5, 0.8, 1])

irr_recomendation = ctrl.Consequent(np.arange(0, 11, 1), 'ir')
irr_recomendation['desnecessaria'] = fuzz.trimf(irr_recomendation.universe, [0, 3, 4])
irr_recomendation['opcional'] = fuzz.trimf(irr_recomendation.universe, [0, 5, 6])
irr_recomendation['recomendada'] = fuzz.trimf(irr_recomendation.universe, [5, 9, 10])

bet_productivity = ctrl.Consequent(np.arange(0, 101, 1), 'bp')
bet_productivity['baixa'] = fuzz.trimf(bet_productivity.universe, [0, 10, 40])
bet_productivity['medio'] = fuzz.trimf(bet_productivity.universe, [0, 50, 70])
bet_productivity['alta'] = fuzz.trimf(bet_productivity.universe, [50, 80, 100])

## Regras fuzzy — Recomendação de Pulverização

Regras do tipo **SE-ENTÃO** que combinam as condições climáticas para classificar a pulverização:

- **Janela disponível** — Delta T ideal, sem chuva significativa, umidade e temperatura ideais.
- **Atenção** — condições parcialmente favoráveis (Delta T aceitável, chuva leve, vento moderado, temperatura amena/quente, ou umidade muito baixa).
- **Proibida** — condições adversas (Delta T crítico/extremo, chuva forte, vento forte/tempestade, temperatura extrema, ou umidade saturada/condensação).

In [9]:
sp_janela_r1 = ctrl.Rule(
    delta_t['ideal_pulv']
    & (rain['seco'] | rain['trace'])
    & humidity['ideal']
    & temperature['ideal'],
    spray_recomendation['janela_disponivel']
)

sp_atencao_r1 = ctrl.Rule(
    (delta_t['aceitavel'] | delta_t['atencao'])
    & rain['trace']
    & (humidity['seco'] | humidity['umido'])
    & (temperature['ameno'] | temperature['quente']),
    spray_recomendation['atencao']
)

sp_atencao_r2 = ctrl.Rule(
    wind['atencao_pulv']
    & (rain['seco'] | rain['trace'])
    & humidity['ideal']
    & temperature['ideal'],
    spray_recomendation['atencao']
)

sp_atencao_r3 = ctrl.Rule(
    rain['leve']
    & (delta_t['ideal_pulv'] | delta_t['aceitavel'])
    & (wind['calmo'] | wind['ideal_pulv'] | wind['moderado']),
    spray_recomendation['atencao']
)

sp_atencao_r4 = ctrl.Rule(
    temperature['frio']
    & (delta_t['ideal_pulv'] | delta_t['aceitavel'])
    & (rain['seco'] | rain['trace']),
    spray_recomendation['atencao']
)

sp_atencao_r5 = ctrl.Rule(
    humidity['deserto'] | humidity['critico_seco'],
    spray_recomendation['atencao']
)

sp_proibida_r1 = ctrl.Rule(
    (delta_t['critico_pulv'] | delta_t['proibido'])
    & (rain['forte'] | rain['muito_forte'] | rain['extrema'] | rain['moderada']),
    spray_recomendation['proibida']
)

sp_proibida_r2 = ctrl.Rule(
    delta_t['inversao_termica'] | delta_t['extremo'],
    spray_recomendation['proibida']
)

sp_proibida_r3 = ctrl.Rule(
    wind['tempestade'] | wind['forte'] | wind['muito_forte'],
    spray_recomendation['proibida']
)

sp_proibida_r4 = ctrl.Rule(
    temperature['frio_extremo'] | temperature['critico'] | temperature['estresse_termico'],
    spray_recomendation['proibida']
)

sp_proibida_r5 = ctrl.Rule(
    humidity['saturado'] | humidity['condensacao'],
    spray_recomendation['proibida']
)

## Regra combinada (múltiplos consequentes)

In [10]:
rule_combined = ctrl.Rule(
    temperature['ideal']
    & humidity['ideal']
    & rain['moderada']
    & wind['moderado']
    & delta_t['aceitavel'],
    (
        water_stress['medio'],
        spray_recomendation['janela_disponivel'],
        irr_recomendation['opcional'],
        bet_productivity['medio']
    )
)

## Sistema de Controle e Simulação

In [11]:
fis_ctrl = ctrl.ControlSystem(
    [
        rule_combined,
        sp_atencao_r1,
        sp_atencao_r2,
        sp_atencao_r3,
        sp_atencao_r4,
        sp_atencao_r5,
        sp_proibida_r1,
        sp_proibida_r2,
        sp_proibida_r3,
        sp_proibida_r4,
        sp_proibida_r5,
        sp_janela_r1,
    ]
)


fis_simulation = ctrl.ControlSystemSimulation(fis_ctrl)

In [19]:
# Janela Disponível
fis_simulation.input['Temperatura'] = 25
fis_simulation.input['Umidade'] = 65
fis_simulation.input['Chuva'] = 0
fis_simulation.input['Vento'] = 8
fis_simulation.input['Delta T'] = 4

fis_simulation.compute()

print(fis_simulation.output['sp'])

3.1174236510531146


In [20]:
# Atenção
fis_simulation.input['Temperatura'] = 32
fis_simulation.input['Umidade'] = 50
fis_simulation.input['Chuva'] = 0
fis_simulation.input['Vento'] = 25
fis_simulation.input['Delta T'] = 10

fis_simulation.compute()

print(fis_simulation.output['sp'])

8.441654135338347


In [21]:
# Proibida 
fis_simulation.input['Temperatura'] = 45
fis_simulation.input['Umidade'] = 95
fis_simulation.input['Chuva'] = 150
fis_simulation.input['Vento'] = 80
fis_simulation.input['Delta T'] = 15

fis_simulation.compute()

print(fis_simulation.output['sp'])

1.4256410256410255
